# Wind speed retrieval

This tutorial applies the inverse method of Harmel & Chami (2012) to synthetic
multidirectional and polarized measurements, see
[Wind speed from sunglint](../wind_speed.rst).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from coxmunk.wind import whitecap_coverage, glint_whitecap, retrieve_wind_speed

## A PARASOL-like acquisition

A ground target is observed in 14 directions along the satellite track, which
crosses the principal plane with an angle of 30°. The Sun is at 30°.

In [ ]:
signed_vza = np.linspace(-55, 55, 14)
geometries = np.array([(30, abs(v), 150 if v >= 0 else 330) for v in signed_vza])

fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
for ws in (3, 8, 13):
    S = glint_whitecap(geometries, ws)
    for k, ax in enumerate(axs):
        ax.plot(signed_vza, S[:, k], 'o-', label=f'{ws} m/s')
for ax, name in zip(axs, 'IQU'):
    ax.set_title(name)
    ax.set_xlabel('viewing zenith angle (deg)')
axs[0].legend()
plt.tight_layout()

At high wind speed, the whitecaps add an unpolarized contribution:

In [ ]:
ws = np.linspace(0, 20, 101)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ws, 100 * whitecap_coverage(ws))
ax.set_xlabel('wind speed (m/s)')
ax.set_ylabel('whitecap coverage (%)')
plt.show()

## Retrieval

Synthetic measurements are generated for a wind speed of 8 m s$^{-1}$ with a
Gaussian noise of $4\times10^{-4}$ (PARASOL noise) and a 5 % relative noise.

In [ ]:
rng = np.random.default_rng(42)
true_ws = 8.
clean = glint_whitecap(geometries, true_ws)
observed = clean + rng.normal(0, 4e-4, clean.shape) + rng.normal(0, 0.05, clean.shape) * clean

res = retrieve_wind_speed(observed, geometries)
print(f'retrieved wind speed: {res.ws:.2f} +/- {res.sigma:.2f} m/s (flag: {res.flag})')

The cost function and the uncertainty criterion $\chi^2(x') \le (1 + \varepsilon)\,\chi^2(x)$
with $\varepsilon = 5$ %:

In [ ]:
x = np.linspace(4, 12, 161)
idx = [0, 1, 2]
chi2 = [np.sum((glint_whitecap(geometries, xi)[:, idx] - observed[:, idx]) ** 2) for xi in x]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(x, chi2)
ax.axhline((1 + 0.05) * res.cost, color='k', ls=':', label=r'$(1+\varepsilon)\,\chi^2_{min}$')
ax.axvspan(res.ws - res.sigma, res.ws + res.sigma, color='orange', alpha=0.3, label=r'$x \pm \sigma$')
ax.axvline(true_ws, color='k', label='true wind speed')
ax.set_xlabel('wind speed (m/s)')
ax.set_ylabel(r'$\chi^2$')
ax.legend()
plt.show()

The uncertainty $\sigma$ describes the flatness of the cost function around its
minimum. Its definition is relative to the value of the minimum, so it is not a
standard deviation of the error.

## Radiance only or radiance and polarization

The retrieval is repeated for several noise realizations, using the
radiance only (`components=('I',)`) or the radiance and the polarization
(default, $I$, $Q$, $U$). The standard deviation of the retrieved wind speeds
measures the precision of each approach.

In [ ]:
n_draws = 15
rows = []
for true_ws in (3., 8., 13.):
    clean = glint_whitecap(geometries, true_ws)
    out = {'I': [], 'IQU': []}
    for _ in range(n_draws):
        noisy = clean + rng.normal(0, 4e-4, clean.shape) + rng.normal(0, 0.05, clean.shape) * clean
        out['I'].append(retrieve_wind_speed(noisy, geometries, components=('I',)).ws)
        out['IQU'].append(retrieve_wind_speed(noisy, geometries).ws)
    rows.append((true_ws, np.mean(out['I']), np.std(out['I']), np.mean(out['IQU']), np.std(out['IQU'])))

print('true   I only (mean, std)   I+Q+U (mean, std)')
for r in rows:
    print(f'{r[0]:>4.0f}   {r[1]:6.2f} {r[2]:6.3f}        {r[3]:6.2f} {r[4]:6.3f}')

In this idealized configuration (no atmosphere, wind direction known), both
approaches retrieve the wind speed without bias, and adding the polarization
slightly reduces the spread of the retrieved values. In the actual
measurements, the polarization also helps separate the sunglint from the
atmospheric and whitecap contributions (see [Polarized sunglint](../polarization.rst)).